# Train, Evaluate, and Track Baseline vs Challenger Models

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
# Build Unity Catalog Volume for MLflow (temporary)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.insurance_claim.mlflow_tmp")
tmp_vol_path = "/Volumes/workspace/insurance_claim/mlflow_tmp"

# Load data from Unity Catalog
train_data = spark.table("workspace.insurance_claim.train_set")
test_data = spark.table("workspace.insurance_claim.test_set")

# Prepare the evaluator
evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

### Baseline Model (Logistic Regression)

In [0]:
# =============================================================================
# 🟢 RUN 1: Baseline Model (Logistic Regression)
# =============================================================================

with mlflow.start_run(run_name="Baseline_LogisticRegression"):
    print("Training Baseline Model...")
    lr = LogisticRegression(featuresCol="features", labelCol="is_fraud", maxIter=10)
    lr_model = lr.fit(train_data)
    
    # Make predictions and evaluate
    lr_predictions = lr_model.transform(test_data)
    lr_auc = evaluator.evaluate(lr_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("ROC_AUC", lr_auc)
    
    # add variable: dfs_tmpdir to point at UC Volume
    mlflow.spark.log_model(lr_model, "baseline_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Baseline ROC AUC: {lr_auc:.4f}")

### Challenger Model (Random Forest)

In [0]:
# =============================================================================
# 🔵 RUN 2: Challenger Model (Random Forest)
# =============================================================================

with mlflow.start_run(run_name="Challenger_RandomForest"):
    print("Training Challenger Model...")
    rf = RandomForestClassifier(featuresCol="features", labelCol="is_fraud", numTrees=50, maxDepth=5)
    rf_model = rf.fit(train_data)
    
    # Make predictions and evaluate
    rf_predictions = rf_model.transform(test_data)
    rf_auc = evaluator.evaluate(rf_predictions)
    
    # Log parameters and metrics
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("ROC_AUC", rf_auc)
    
    # add variable: dfs_tmpdir to pint at UC Volume
    mlflow.spark.log_model(rf_model, "challenger_model", dfs_tmpdir=tmp_vol_path)
    
    print(f"✅ Challenger ROC AUC: {rf_auc:.4f}")